[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/00-setup/notebooks/03-colab-and-requirements.ipynb)

# 00 · Notebook 3 — Colab tour and `requirements.txt`

**Goal:** practise the Colab features from the lesson, and learn how a `requirements.txt` file pins versions.

This notebook runs anywhere. Colab-only cells check first whether they are in Colab and explain what they would do otherwise.

For the GPU part in Colab: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**, then run all cells. Locally you will see "CPU only", which is fine.

## 1. Where am I running?

In [1]:
import sys, os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("In Colab:", IN_COLAB)
print("Current folder:", Path.cwd())

In Colab: False
Current folder: /home/claude/AI-Engineer/00-setup/notebooks


## 2. Is there a GPU?

A **GPU** (graphics processing unit) does many maths operations at once and makes deep learning much faster. PyTorch (a deep learning library, module 05) can tell us if one is visible. On Colab with a T4 runtime this prints the GPU name.

In [2]:
try:
    import torch
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("CPU only (in Colab: Runtime -> Change runtime type -> T4 GPU)")
except ImportError:
    print("PyTorch not installed")

CPU only (in Colab: Runtime -> Change runtime type -> T4 GPU)


In Colab you can also run `!nvidia-smi`, a terminal command that shows the GPU, its memory and what is using it. We only run it when a GPU exists, because elsewhere the command is not found.

In [3]:
import shutil
if shutil.which("nvidia-smi"):
    !nvidia-smi
else:
    print("nvidia-smi not available here (no NVIDIA GPU).")

nvidia-smi not available here (no NVIDIA GPU).


## 3. Mounting Google Drive

**Mounting** connects a storage drive so it appears as a folder. Colab's disk is wiped when the runtime disconnects, so save anything important to Drive.

In Colab, the cell asks you to choose your Google account and allow access. Then your files are under `/content/drive/MyDrive/`. Outside Colab it only explains.

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/ai-course")
    save_dir.mkdir(parents=True, exist_ok=True)
    print("Save long-lived files in:", save_dir)
else:
    print("Not in Colab. Locally your files already persist on disk.")

Not in Colab. Locally your files already persist on disk.


## 4. Uploading and downloading files

`files.upload()` shows a **Choose Files** button under the cell. The chosen file lands in `/content/`. `files.download(name)` sends a file back to your computer. You can also use the **folder icon → Upload** button in Colab's left sidebar.

To keep the notebook runnable top to bottom, the upload line is commented out. Remove the `#` in Colab to try it.

In [5]:
sample = Path("sample_results.csv")
sample.write_text("student,score\nAsha,88\nRavi,92\n", encoding="utf-8")
print("Created", sample, "with", len(sample.read_text().splitlines()), "lines")

if IN_COLAB:
    from google.colab import files
    # uploaded = files.upload()          # remove the # to try uploading
    # files.download(str(sample))        # remove the # to download the CSV
    print("In Colab: un-comment the lines above to try upload/download.")
else:
    print("Not in Colab: the file is simply on your disk at", sample.resolve())

Created sample_results.csv with 3 lines
Not in Colab: the file is simply on your disk at /home/claude/AI-Engineer/00-setup/notebooks/sample_results.csv


## 5. Colab Secrets

Click the **key icon** in Colab's left sidebar → **+ Add new secret** → Name `OPENAI_API_KEY`, Value = your key → switch on **Notebook access**. Then `userdata.get` reads it. The value is stored in your Google account, not in the notebook.

In [6]:
if IN_COLAB:
    from google.colab import userdata
    try:
        key = userdata.get("OPENAI_API_KEY")
        print("OPENAI_API_KEY found in Colab Secrets:", key is not None)
    except Exception as err:
        print("Secret not available:", type(err).__name__)
else:
    print("Not in Colab: use a .env file instead (see notebook 2).")

Not in Colab: use a .env file instead (see notebook 2).


## 6. `requirements.txt`: the shopping list

A **requirements.txt** lists the packages a project needs, one per line. **Pinning** with `==` fixes an exact version so the project behaves the same on every machine.

We write a small example file, then read it back line by line.

In [7]:
req = Path("example-requirements.txt")
req.write_text(
    "# example requirements file\n"
    "numpy==2.1.3\n"
    "pandas>=2.0,<4\n"
    "python-dotenv\n",
    encoding="utf-8",
)
for line in req.read_text().splitlines():
    print(line)

# example requirements file
numpy==2.1.3
pandas>=2.0,<4
python-dotenv


## 7. Which lines are pinned, and do they match what is installed?

We split each line into the package name and the version rule, then compare with what is installed. An unpinned line (no version) is flagged, because it can install a different version next month.

In [8]:
import re
import importlib.metadata as meta

for line in req.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    name = re.split(r"[=<>~!]", line)[0].strip()
    rule = line[len(name):] or "(no version - UNPINNED)"
    try:
        installed = meta.version(name)
    except meta.PackageNotFoundError:
        installed = "not installed"
    exact = rule.startswith("==")
    match = exact and rule[2:] == installed
    note = "matches" if match else ("differs" if exact else "")
    print(f"{name:<14} wants {rule:<24} installed {installed:<12} {note}")

numpy          wants ==2.1.3                  installed 2.5.3        differs
pandas         wants >=2.0,<4                 installed 3.0.5        
python-dotenv  wants (no version - UNPINNED)  installed 1.2.3        


## 8. `pip freeze` style output

`pip freeze` prints every installed package as `name==version`. Below we produce the first few lines the same way using `importlib.metadata`, so the notebook does not modify anything. In a terminal you would run `python -m pip freeze > requirements.txt`.

In [9]:
dists = sorted({(d.metadata["Name"], d.version) for d in meta.distributions() if d.metadata["Name"]},
               key=lambda pair: pair[0].lower())
for name, version in dists[:8]:
    print(f"{name}=={version}")
print("...", len(dists), "packages in total")

absl-py==2.5.1
accelerate==1.16.0
altair==6.3.0
annotated-doc==0.0.7
annotated-types==0.8.0
anthropic==2.12.0
anyio==4.15.1
argcomplete==3.1.4
... 302 packages in total


## 9. Clean up

In [10]:
for f in [sample, req]:
    f.unlink(missing_ok=True)
print("Removed practice files.")

Removed practice files.


## Summary

- Colab: choose a **T4 GPU** runtime for deep learning; check with `torch.cuda.is_available()` or `!nvidia-smi`.
- The Colab disk is temporary: mount **Drive** or download your results.
- Upload with the Files panel or `files.upload()`; store keys in the **Secrets** panel.
- `requirements.txt` with `==` pins versions; `pip freeze > requirements.txt` records your environment; `pip install -r requirements.txt` rebuilds it.